# Reconstructed PM2.5, aerosol optical depth, and meteorology records for South Tangerang City, Indonesia, 2005–2025 Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This walkthrough emphasizes how to:
- load Croissant metadata from a URL,
- inspect record sets and fields using their `@id` values,
- extract records into pandas DataFrames,
- perform basic data cleaning and exploratory analysis,
- create simple visualizations for numeric and temporal patterns.

> Note: This dataset package contains rich descriptive metadata. The notebook is written defensively so it can still explore the package even if some record sets are unavailable or populated dynamically.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.b51g-0645/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print('Dataset loaded successfully.')
print(f'Name: {getattr(metadata, "name", "N/A")}')
print(f'Description: {getattr(metadata, "description", "N/A")}')
print(f'Identifier: {getattr(metadata, "identifier", "N/A")}')
print(f'License: {getattr(metadata, "license", "N/A")}')
print(f'Version: {getattr(metadata, "version", "N/A")}')

In [ ]:
# Convert metadata to a JSON-like object for structured inspection when needed
metadata_json = metadata.to_json()

print('Top-level metadata keys:')
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities below are referenced by their Croissant `@id` values whenever available.

In [ ]:
# Inspect high-level dataset metadata from the JSON representation
dataset_id = metadata_json.get('@id')
dataset_name = metadata_json.get('name')
record_sets_json = metadata_json.get('recordSet', [])
distributions_json = metadata_json.get('distribution', [])

print(f'Dataset @id: {dataset_id}')
print(f'Dataset name: {dataset_name}')
print(f'Number of distributions: {len(distributions_json)}')
print(f'Number of record sets declared: {len(record_sets_json)}')

In [ ]:
# Display declared record sets and fields using @id references
if not record_sets_json:
    print('No record sets are explicitly declared in the metadata JSON under `recordSet`.')
    print('This package may primarily expose distributions and descriptive metadata, or record sets may be resolved differently by the backend.')
else:
    for rs in record_sets_json:
        rs_id = rs.get('@id', 'N/A')
        rs_name = rs.get('name', 'N/A')
        print(f'\nRecord set @id: {rs_id}')
        print(f'  name: {rs_name}')
        fields = rs.get('field', [])
        print(f'  number of fields: {len(fields)}')
        for field in fields:
            field_id = field.get('@id', 'N/A')
            field_name = field.get('name', 'N/A')
            field_type = field.get('dataType', 'N/A')
            source = field.get('source', {})
            column_id = source.get('extract', {}).get('column') if isinstance(source, dict) else None
            print(f'    - field @id: {field_id} | name: {field_name} | dataType: {field_type} | column @id: {column_id}')

In [ ]:
# Show declared file/distribution objects using @id
print('Declared distributions:')
for i, dist in enumerate(distributions_json, start=1):
    if isinstance(dist, dict):
        print(f'{i}. distribution @id: {dist.get("@id", "N/A")}')
    else:
        print(f'{i}. distribution: {dist}')

### Optional record preview
If record sets are available through `mlcroissant`, the next cell will preview a few records from each record set using its `@id`.

In [ ]:
record_set_ids = [rs.get('@id') for rs in record_sets_json if isinstance(rs, dict) and rs.get('@id')]
print('Record set @id values found:', record_set_ids)

if not record_set_ids:
    print('No record set @id values available to preview.')
else:
    for record_set_id in record_set_ids:
        print(f'\nPreviewing record set: {record_set_id}')
        try:
            for idx, x in enumerate(dataset.records(record_set=record_set_id)):
                print(x)
                if idx >= 2:
                    break
        except Exception as e:
            print(f'Could not load records for {record_set_id}: {e}')

## 3. Data Extraction
Load data from available record sets into DataFrames for analysis. Record sets and fields should always be referenced by their `@id` values.

In [ ]:
# Extract data from each available record set
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f'Loaded {len(records)} records from record set @id: {record_set_id}')
    except Exception as e:
        print(f'Failed to load record set @id {record_set_id}: {e}')

print('\nLoaded DataFrame keys:')
print(list(dataframes.keys()))

In [ ]:
# Inspect the first successfully loaded DataFrame, if any
if dataframes:
    first_record_set_id = next(iter(dataframes))
    print(f'Using record set @id: {first_record_set_id}')
    print('Columns:')
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print('No record sets were successfully loaded into DataFrames.')

### Build a field dictionary from metadata
This helper structure maps each record set `@id` to its field `@id` values and other useful metadata. It is especially helpful when datasets define many fields.

In [ ]:
field_dictionary = {}

for rs in record_sets_json:
    if not isinstance(rs, dict):
        continue
    rs_id = rs.get('@id')
    fields = rs.get('field', [])
    field_dictionary[rs_id] = []
    for field in fields:
        if not isinstance(field, dict):
            continue
        source = field.get('source', {}) if isinstance(field.get('source', {}), dict) else {}
        extract = source.get('extract', {}) if isinstance(source.get('extract', {}), dict) else {}
        field_dictionary[rs_id].append({
            'field_@id': field.get('@id'),
            'field_name': field.get('name'),
            'dataType': field.get('dataType'),
            'column_@id': extract.get('column')
        })

for rs_id, fields in field_dictionary.items():
    print(f'\nRecord set @id: {rs_id}')
    display(pd.DataFrame(fields))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. The code below automatically identifies numeric and categorical fields when possible so the notebook remains reusable.

In [ ]:
import numpy as np

if not dataframes:
    print('EDA skipped because no tabular record sets were loaded.')
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    print(f'Running EDA on record set @id: {record_set_id}')
    print(f'Data shape: {df.shape}')
    
    # Attempt numeric conversion where reasonable
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='ignore')
    
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()
    
    print('Numeric columns detected:')
    print(numeric_columns)
    print('\nNon-numeric columns detected:')
    print(non_numeric_columns)

In [ ]:
if not dataframes:
    print('No data available for filtering and normalization.')
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='ignore')
    
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    
    if not numeric_columns:
        print('No numeric fields found for threshold filtering.')
    else:
        numeric_field = numeric_columns[0]
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        
        print(f'Selected numeric field: {numeric_field}')
        print(f'Using threshold (median): {threshold}')
        print(f'Filtered records with {numeric_field} > {threshold}:')
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field].std()
        if pd.notnull(std_value) and std_value != 0:
            normalized_col = f'{numeric_field}_normalized'
            filtered_df[normalized_col] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
            )
            print(f'Normalized {numeric_field} for filtered records:')
            display(filtered_df[[numeric_field, normalized_col]].head())
        else:
            print(f'Skipping normalization because the standard deviation of {numeric_field} is zero or undefined.')

In [ ]:
if not dataframes:
    print('No data available for grouping analysis.')
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='ignore')
    
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    group_candidates = [c for c in df.columns if c not in numeric_columns]
    
    if numeric_columns and group_candidates:
        numeric_field = numeric_columns[0]
        group_field = group_candidates[0]
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        grouped_df = filtered_df.groupby(group_field)[numeric_columns].mean(numeric_only=True)
        print(f'Grouped filtered data by {group_field}:')
        display(grouped_df.head())
    else:
        print('Grouping skipped because suitable numeric and categorical fields were not both available.')

In [ ]:
# Basic missing-value summary
if not dataframes:
    print('No data available for missing-value analysis.')
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id]
    missing_summary = df.isna().sum().sort_values(ascending=False).to_frame('missing_count')
    missing_summary['missing_fraction'] = missing_summary['missing_count'] / len(df) if len(df) else 0
    print(f'Missing-value summary for record set @id: {record_set_id}')
    display(missing_summary.head(20))

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

if not dataframes:
    print('Visualization skipped because no tabular data was loaded.')
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='ignore')
    
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    
    if numeric_columns:
        numeric_field = numeric_columns[0]
        plt.figure(figsize=(8, 5))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field}\n(record set @id: {record_set_id})')
        plt.xlabel(numeric_field)
        plt.ylabel('Count')
        plt.show()
    else:
        print('No numeric columns available for histogram plotting.')

In [ ]:
if not dataframes:
    print('Scatter plot skipped because no data was loaded.')
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='ignore')
    
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    
    if len(numeric_columns) >= 2:
        x_field = numeric_columns[0]
        y_field = numeric_columns[1]
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=x_field, y=y_field, alpha=0.7)
        plt.title(f'{y_field} vs {x_field}\n(record set @id: {record_set_id})')
        plt.xlabel(x_field)
        plt.ylabel(y_field)
        plt.show()
    else:
        print('At least two numeric columns are required for a scatter plot.')

In [ ]:
# Optional time-series plot if a date-like field exists
if not dataframes:
    print('Time-series visualization skipped because no data was loaded.')
else:
    record_set_id = next(iter(dataframes))
    df = dataframes[record_set_id].copy()
    
    date_candidates = [c for c in df.columns if 'date' in str(c).lower() or 'time' in str(c).lower() or 'year' in str(c).lower()]
    numeric_columns = []
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='ignore')
    numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    
    if date_candidates and numeric_columns:
        date_field = date_candidates[0]
        value_field = numeric_columns[0]
        temp_df = df[[date_field, value_field]].copy()
        temp_df[date_field] = pd.to_datetime(temp_df[date_field], errors='coerce')
        temp_df = temp_df.dropna(subset=[date_field, value_field]).sort_values(date_field)
        
        if not temp_df.empty:
            plt.figure(figsize=(10, 5))
            plt.plot(temp_df[date_field], temp_df[value_field])
            plt.title(f'{value_field} over {date_field}\n(record set @id: {record_set_id})')
            plt.xlabel(date_field)
            plt.ylabel(value_field)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        else:
            print('A date-like field was found, but no valid rows remained after parsing.')
    else:
        print('No suitable date-like and numeric field combination found for time-series plotting.')

## 6. Conclusion
This notebook demonstrated how to load a Croissant dataset with `mlcroissant`, inspect metadata, identify record sets and fields using their `@id` values, and extract available records into pandas DataFrames for exploration.

Key takeaways:
- The dataset metadata provides rich descriptive context, including provenance, limitations, and usage guidance.
- Record sets, fields, and source columns should be referenced by their Croissant `@id` values whenever available.
- The notebook is written to adapt to datasets that may expose record sets directly, indirectly, or only through distribution metadata.
- Once records are loaded, common EDA tasks such as filtering, normalization, grouping, missing-value inspection, and plotting can be applied in a reusable way.

For deeper analysis, you can extend this notebook by:
- selecting specific record set `@id` values once they are available,
- mapping field `@id` values to domain variables such as PM2.5, AOD, meteorology, space, and time,
- joining multiple record sets together,
- building forecasting or reconstruction models from the extracted tables.